# Streaming chat and cancellation

[Back to the introduction](ch01_demo.ipynb). Run in order in a fresh kernel. This loads SmolLM2-135M-Instruct (about 140 MB, WASM), a small instruction-tuned model: answers are short and not always accurate. See [Streaming and Cancellation](https://github.com/decitre/lmlite/blob/main/docs/USAGE.md#streaming-and-cancellation) for the chunk format, concurrency and backend limits.

In [ ]:
import asyncio

from lmlite import LLM

In [ ]:
chat_llm = await LLM.create(
    generator_model="HuggingFaceTB/SmolLM2-135M-Instruct",
    backend="wasm",
    max_new_tokens=120,
    do_sample=False,
)
question = [{"role": "user", "content": "Explain embeddings simply."}]

`chat_stream()` yields text chunks as they are generated. Joined and stripped, they are the `chat()` answer.

In [ ]:
chunks = []
async for chunk in chat_llm.chat_stream(question):
    print(chunk, end="", flush=True)
    chunks.append(chunk)
print(f"\n\n{len(chunks)} chunks")

Leaving the loop early (`break`, an error or a kernel interrupt) cancels the generation, and the instance stays usable.

In [ ]:
async for chunk in chat_llm.chat_stream(question):
    print(chunk, end="", flush=True)
    if chunk.rstrip().endswith("."):
        break  # Stop after the first sentence.
print("\n[stopped]")

A cell blocks while it streams, so `cancel()` is called from code running alongside the stream, here an `asyncio` task. It stops generation after the current token and returns `False` if nothing was running; the stream then ends without error.

In [ ]:
first_chunk = asyncio.Event()


async def collect():
    partial = []
    async for chunk in chat_llm.chat_stream(question):
        partial.append(chunk)
        first_chunk.set()
    return partial


task = asyncio.create_task(collect())
await first_chunk.wait()
print("Cancelled:", await chat_llm.cancel())
print("Partial answer:", "".join(await task).strip())
print("Nothing left to cancel:", not await chat_llm.cancel())

In [ ]:
await chat_llm.dispose()

The model is disposed without clearing cached downloads. Rerun the notebook to load it again from the cache.